# KPMP TAL Subset Analysis Notebook

In this notebook, we will walk through the process of downloading and loading the full KPMP kidney atlas, subsetting to Thick Ascending Limb (TAL) cells, computing a dedicated UMAP embedding for that subset, and visualizing key biological annotations. We’ll also demonstrate how to refine that embedding to focus on merged TAL subclasses.


In [ ]:
import logging
from pathlib import Path

import re
import requests
import scanpy as sc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

In [ ]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
logger = logging.getLogger(__name__)
logger.info("Starting KPMP TAL analysis notebook")

## 1. Data Download and Citation

We will use the CZI CellxGene Consortium’s 2023 _Integrated Single-nucleus and Single-cell RNA-seq of the Adult Human Kidney_ dataset as our starting point. The following function handles reliable downloading, with a progress bar and automatic retry via a temporary file.

> **Data source:**  
> CZI CellxGene Consortium. (2023). _Integrated Single-nucleus and Single-cell RNA-seq of the Adult Human Kidney_ [Dataset]. CZ CellxGene. Retrieved from https://cellxgene.cziscience.com/collections/bcb61471-2a44-4d00-a0af-ff085512674c


In [ ]:
def download(url: str, path: Path) -> None:
    """
    Download with progress bar and retry-safe temp file.

    Args:
        url (str): URL to download from.
        path (Path): Destination file path.
    """
    from tqdm.autonotebook import tqdm

    temp_path = path.with_suffix(path.suffix + ".tmp")
    logger.info(f"Downloading {url} → {path.name}")

    try:
        with requests.Session() as session:
            session.headers.update({"User-Agent": "Mozilla/5.0"})
            response = session.get(url, stream=True, timeout=30)
            response.raise_for_status()
            total = int(response.headers.get("content-length", 0))

            with tqdm(total=total, unit="B", unit_scale=True, desc=path.name) as pbar:
                with open(temp_path, "wb") as f:
                    for chunk in response.iter_content(chunk_size=65536):
                        if chunk:
                            f.write(chunk)
                            pbar.update(len(chunk))
        temp_path.replace(path)
        logger.info(f"Download complete: {path.name}")
    except Exception as e:
        temp_path.unlink(missing_ok=True)
        logger.error(f"Download failed: {e}")
        raise

In [ ]:
kpmp_path = Path("data/kpmp.h5ad")
kpmp_path.parent.mkdir(exist_ok=True)
if not kpmp_path.exists():
    download(
        "https://datasets.cellxgene.cziscience.com/45a06603-f923-45af-b4c3-8ead77aa2e78.h5ad",
        kpmp_path,
    )

## 2. Load and Subset to TAL Cells

Once the atlas file is present (or successfully downloaded), we load it into an AnnData object. We then subset to only those cells annotated as "TAL" in the `subclass.l1` column. A quick log message confirms the number of TAL cells retained.


In [ ]:
logger.info("Reading full KPMP atlas")
adata = sc.read(kpmp_path)
adata

In [ ]:
logger.info(f"Data range: {adata.X.min()} - {adata.X.max()}")

In [ ]:
logger.info("Subsetting to TAL (subclass.l1 == 'TAL')")
adata_tal = adata[adata.obs["subclass.l1"] == "TAL"].copy()
logger.info(f"TAL subset: {adata_tal.n_obs} cells")

In [ ]:
logger.info(f"Data range for TAL: {adata_tal.X.min()} - {adata_tal.X.max()}")

In [ ]:
sc.pl.umap(
    adata_tal,
    color="subclass.l2",
    frameon=False,
    title="TAL subset UMAP",
    save="tal_umap.png",
)

## 3. Preprocessing for TAL-only UMAP

To generate a high-resolution UMAP for the TAL subset, we follow the standard Scanpy pipeline:

1. **Total-count normalization** and **log<sub>1p</sub> transformation** to bring all cells onto a common scale.
2. **Highly variable gene (HVG) selection** to focus dimensionality reduction on the 2,000 most informative genes within TAL.
3. **Scaling** each gene to unit variance, clipping extreme values to ±10 to limit the influence of outliers.
4. **Principal component analysis (PCA)**, retaining 50 components to capture major variance axes.
5. **Neighbor graph construction** (30 neighbors, cosine metric) to define local cell neighborhoods.
6. **UMAP embedding** (min_dist = 0.3) to project cells into two dimensions for visualization.

Each of these steps is accompanied by an INFO log entry so we can monitor runtime and troubleshoot if necessary.


In [ ]:
logger.info("Normalizing and log1p transforming")

# We will work with raw expression data
adata_tal.X = adata_tal.raw.X.copy()

logger.info("Normalizing and log1p transforming from raw counts")
sc.pp.normalize_total(adata_tal, target_sum=1e4)
sc.pp.log1p(adata_tal)

In [ ]:
logger.info(f"Data range for TAL after normalization: {adata_tal.X.min()} - {adata_tal.X.max()}")

In [ ]:
# logger.info("Selecting highly-variable genes (n_top_genes=2000)")
# sc.pp.highly_variable_genes(
#     adata_tal,
#     flavor="seurat_v3",
#     n_top_genes=2000,
#     batch_key=None,
# )
# adata_tal = adata_tal[:, adata_tal.var.highly_variable].copy()

In [ ]:
logger.info("Scaling data (clipping at |10|)")
sc.pp.scale(adata_tal, max_value=10)

In [ ]:
logger.info("Computing PCA (50 comps) and neighbor graph (n_neighbors=30)")
sc.tl.pca(adata_tal, n_comps=50, svd_solver="arpack")
sc.pp.neighbors(adata_tal, n_neighbors=30, n_pcs=50, metric="cosine")

In [ ]:
logger.info("Computing UMAP (min_dist=0.3)")
sc.tl.umap(adata_tal, min_dist=0.3)

In [ ]:
logger.info("Plotting UMAP colored by subclass.l2, disease, tissue")
titles = {
    "subclass.l2": "Cell subtype",
    "disease": "Disease state",
    "tissue": "Tissue origin",
}

sc.pl.umap(
    adata_tal,
    color=list(titles.keys()),
    title=list(titles.values()),
    ncols=3,
    wspace=0.4,
    frameon=False,
    save="__tal_standard_umap.png",
)

In [ ]:
tal_out = Path("data/kpmp_tal_preprocessed.h5ad")

In [ ]:
logger.info(f"Writing preprocessed TAL AnnData → {tal_out}")
adata_tal.write(tal_out, compression="gzip")

## 4. High-Resolution TAL-Subclass Embedding

TAL epithelium splits into several finer subclasses (e.g. C-TAL, M-TAL, aTAL1, aTAL2). To focus on these, we:

1. **Merge** the raw `subclass.l2` annotations into the four categories of interest.
2. **Filter** out any cells not mapping to one of those categories.
3. **Recompute** scaling, PCA, neighbor graph, and UMAP—this time optimized for only those TAL subclasses.

This "zoomed-in" embedding uncovers substructure that the global TAL UMAP might underweight.

In [ ]:
adata_tal = sc.read(tal_out)

In [ ]:
adata_tal

In [ ]:
logger.info("Merging subclass.l2 into categories C-TAL, M-TAL, aTAL")
mapping = {"C-TAL": "C-TAL", "M-TAL": "M-TAL", "aTAL1": "aTAL", "aTAL2": "aTAL"}

merged = adata_tal.obs["subclass.l2"].map(mapping)
categories = list(dict.fromkeys(mapping.values()))

adata_tal.obs["merged_subclass"] = pd.Categorical(
    merged,
    categories=categories,
    ordered=True
)

In [ ]:
adata_filtered = adata_tal[adata_tal.obs["merged_subclass"].notna()].copy()
logger.info(f"Filtered retention: {adata_filtered.n_obs} TAL cells")

In [ ]:
logger.info(f"Data range for filtered TAL: {adata_filtered.X.min()} - {adata_filtered.X.max()}")

In [ ]:
adata_filtered.X = adata_filtered.raw.X.copy()
logger.info(f"Data range for filtered TAL before normalization: {adata_filtered.X.min()} - {adata_filtered.X.max()}")

In [ ]:
logger.info("Normalizing and log1p transforming from raw counts")
sc.pp.normalize_total(adata_filtered, target_sum=1e4)
sc.pp.log1p(adata_filtered)

In [ ]:
logger.info(f"Data range after normalization: {adata_filtered.X.min()} - {adata_filtered.X.max()}")

In [ ]:
logger.info("Recomputing PCA, neighbors, UMAP on merged_subclass subset")
sc.pp.scale(adata_filtered, max_value=10)

In [ ]:
sc.tl.pca(adata_filtered, n_comps=50, svd_solver="arpack")
sc.pp.neighbors(adata_filtered, n_neighbors=30, n_pcs=50, metric="cosine")

In [ ]:
sc.tl.umap(adata_filtered, min_dist=0.3)

In [ ]:
logger.info("Plotting merged_subclass UMAP")
sc.pl.umap(
    adata_filtered,
    color="merged_subclass",
    title="TAL subclasses (C-TAL, M-TAL, aTAL)",
    frameon=False,
    save="_kpmp_tal_umap_merged_subclass.png",
)

In [ ]:
sc.pl.umap(
    adata_filtered,
    color="tissue",
    title="TAL tissue origin (merged_subclass)",
    frameon=False,
    save="_kpmp_tal_umap_tissue_merged_subclass.png",
)

In [ ]:
filtered_out = Path("data/kpmp_tal_filtered.h5ad")
logger.info(f"Writing merged-subset TAL AnnData → {filtered_out}")
adata_filtered.write(filtered_out, compression="gzip")